# Vérification de fake news par la knowledge-base method

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r drive/MyDrive/Projet_NLP.

Mounted at /content/drive
cp: missing destination file operand after 'drive/MyDrive/Projet_NLP.'
Try 'cp --help' for more information.


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import torch
device = 'GPU' if torch.cuda.is_available() else 'CPU'
print('Vous utilisez le {} '.format(device))

Vous utilisez le CPU 


In [ ]:
!pip install transformers torch spacy pandas

In [ ]:
import requests

url = "https://zenodo.org/record/3609356/files/groundtruth.csv?download=1"
response = requests.get(url, allow_redirects=True)

with open("groundtruth.csv", "wb") as f:
    f.write(response.content)
print("✅ Fichier groundtruth.csv téléchargé avec succès !")

✅ Fichier groundtruth.csv téléchargé avec succès !


In [ ]:
import pandas as pd

dataset = pd.read_csv("groundtruth.csv")
print(dataset.head())
print(dataset.columns.to_list)
print(dataset['Verdict'].value_counts())
print(dataset.shape)

# On crée une colonne 'labels' binaire : 1 pour le niveau 1, 0 pour le reste
dataset['labels'] = dataset['Verdict'].apply(lambda x: 1 if x == 1 else 0)

# On ne garde que le texte et le nouveau label
dataset = dataset[['Text', 'labels']]

   Sentence_id                                               Text  \
0           26      You know, I saw a movie - "Crocodile Dundee."   
1           80  We're consuming 50 percent of the world's coca...   
2          129   That answer was about as clear as Boston harbor.   
3          131                          Let me help the governor.   
4          172  We've run up more debt in the last eight years...   

           Speaker   Speaker_title Speaker_party         File_id  Length  \
0      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       9   
1  Michael Dukakis        Governor      DEMOCRAT  1988-09-25.txt       8   
2      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       9   
3      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       5   
4  Michael Dukakis        Governor      DEMOCRAT  1988-09-25.txt      22   

   Line_number  Sentiment  Verdict  
0           26   0.000000        0  
1           80  -0.740979        1  
2          129   

In [ ]:
dataset.head()

,Text,labels
0,"You know, I saw a movie - ""Crocodile Dundee.""",0
1,We're consuming 50 percent of the world's coca...,1
2,That answer was about as clear as Boston harbor.,0
3,Let me help the governor.,0
4,We've run up more debt in the last eight years...,1


In [ ]:
from datasets import Dataset, DatasetDict

# On transforme le DataFrame en Dataset Hugging Face
hg_dataset = Dataset.from_pandas(dataset[['Text', 'labels']])

# on divise notre dataset en train/test

split_dataset = hg_dataset.train_test_split(test_size=0.2)

# Partie Claim detecion

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # L'option batched=True envoie une liste de textes ici
    return tokenizer(examples["Text"], padding="max_length", truncation=True)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Lancer l'entraînement
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/825 [00:00<?, ? examples/s]

Map:   0%|          | 0/207 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./claim_detector_model")
tokenizer.save_pretrained("./claim_detector_model")

NameError: name 'trainer' is not defined

In [ ]:
from transformers import pipeline

claim_pipeline = pipeline("text-classification", model="./claim_detector_model", tokenizer="./claim_detector_model", device=0)


def detect_claim(text, threshold=0.2):
    results = claim_pipeline(text)
    # Dans ton modèle, vérifie si LABEL_1 est bien le "Check-worthy"
    # Si le score pour LABEL_1 est > 0.2, on y va.
    score_claim = next((r['score'] for r in results if r['label'] == 'LABEL_1'), 0)
    return score_claim > threshold, score_claim

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './claim_detector_model'.

In [ ]:
! pip install wikipedia-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 62.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 35.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 28.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('

In [ ]:
! ls

claim_detector_model  drive  groundtruth.csv  results  sample_data


In [ ]:
%cd /content/drive/MyDrive/Projet_NLP/knowledge_branch

/content/drive/MyDrive/Projet_NLP/knowledge_branch


In [ ]:
! ls

claim_detector_model   evidence_retrieval.py  knowledge.ipynb  results
claim_verification.py  groundtruth.csv	      __pycache__


In [ ]:
from evidence_retrieval import EvidenceRetriever

# Remplace par tes vraies clés obtenues sur les portails
WOLFRAM_APPID = "LEU7Y6728T"
GOOGLE_API_KEY = None
GOOGLE_CSE_ID = "151bf4aa4eae44373"

# Initialisation du nouveau Retriever
retriever = EvidenceRetriever(
    google_api_key=GOOGLE_API_KEY,
    google_cse_id=GOOGLE_CSE_ID,
    wolfram_app_id=WOLFRAM_APPID,
    config_languages={'en': 'en_core_web_sm', 'fr': 'fr_core_news_sm', 'es': 'es_core_news_sm'}
)

test_claims = {
    "The Eiffel Tower is 330 meters tall": "en",
    "Le taux de chômage en France est de 7%": "fr",
    "La capital de España es Madrid": "es"
}

print("--- Lancement du Test Multi-Sources ---\n")

for claim, language in test_claims.items():
    # On utilise la méthode qui gère les priorités
    evidence = retriever.get_evidence(claim, language)

    if evidence:
        source_name = evidence.get('title', 'Inconnue')
        print(f"✅ Trouvé via [{source_name}] ({language})")
        print(f"Phrase : {claim}")
        print(f"Preuve : {evidence['content'][:150]}...")
        print(f"Lien : {evidence.get('url', 'N/A')}\n")
    else:
        print(f"❌ Aucune preuve trouvée pour : {claim}\n")

--- Lancement du Test Multi-Sources ---



JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# Partie Claim Verification

In [ ]:
from claim_verification import ClaimVerifier

# On initialise le verifier

claimverifier = ClaimVerifier()
claimverifier.verify("New York is in the United States", "New York is one of the bigest city of the United States")

Chargement du modèle de vérification


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

('SUPPORTED', 0.9993575215339661)

In [ ]:
# --- CONFIGURATION ---

verifier = ClaimVerifier()
text_to_check = "Tom Cruise was born in 1962"
FORCE_CHECK = True # Pour forcer la vérification même si BERT hésite

is_claim, detection_score = detect_claim(text_to_check, threshold=0.2)

if is_claim or FORCE_CHECK:
    if not is_claim:
        print(f"⚠️ Détection faible ({detection_score:.2f}), mais on vérifie quand même...")
    else:
        print(f"🔍 Claim détecté ! (Confiance: {detection_score:.2f})")

    # ÉTAPE 2 : Récupération
    evidence = retriever.get_evidence(text_to_check, language='en')

    if evidence:
        # ÉTAPE 3 : Vérification
        verdict, confidence = verifier.verify(text_to_check, evidence['content'])
        print(f"\n--- RÉSULTAT FINAL ---")
        print(f"Phrase : {text_to_check}")
        print(f"Verdict : {verdict} (Score NLI: {confidence:.2f})")
        print(f"Source : {evidence['url']} (via {evidence.get('title')})")
    else:
        print("❌ Aucune preuve trouvée.")
else:
    print("ℹ️ Cette phrase n'a pas été jugée digne d'intérêt par le détecteur.")

In [ ]:
retriever = EvidenceRetriever(google_api_key=GOOGLE_API_KEY,
    google_cse_id="151bf4aa4eae44373",
    wolfram_app_id="LEU7Y6728T",
    config_languages={'en': 'en_core_web_sm', 'fr': 'fr_core_news_sm', 'es': 'es_core_news_sm'})

verifier = ClaimVerifier()

def process_full_text(text, language='en'):
    nlp = retriever.nlp_models.get(language)
    doc = nlp(text)
    final_report = []

    for sent in doc.sents:
        sentence = sent.text.strip()
        if not sentence: continue

        # --- DÉTECTION HYBRIDE ---
        is_claim, detection_score = detect_claim(sentence, threshold=0.2)
        # On force la vérification si l'IA dit OUI ou si spaCy voit des entités
        has_entities = len([ent for ent in sent.ents if ent.label_ in ['GPE', 'PERSON', 'DATE', 'ORG']])

        if is_claim or has_entities:
            print(f"🔎 Analyse de : {sentence} (Score IA: {detection_score:.2f})")

            # --- RÉCUPÉRATION (Appel crucial) ---
            evidence = retriever.get_evidence(sentence, language)

            if evidence:
                # --- VÉRIFICATION ---
                verdict, score = verifier.verify(sentence, evidence['content'])
                final_report.append({
                    "phrase": sentence,
                    "verdict": verdict,
                    "confiance": score,
                    "source": evidence['url'],
                    "source_name": evidence.get('title')
                })
            else:
                final_report.append({"phrase": sentence, "verdict": "NO EVIDENCE FOUND"})
        else:
            print(f"⏭️ Ignoré : {sentence}")

    return final_report

Chargement du modèle de vérification


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
mon_article = """
Tom Cruise is an american actor. The Eiffel Tower is in Paris. it was built in 1990.

"""

resultats = process_full_text(mon_article, language='en')

# Affichage propre
for r in resultats:
    print(f"[{r['verdict']}] {r['phrase']} (Source: {r.get('source', 'N/A')})")

🔎 Analyse de : Tom Cruise is an american actor.
🔎 Analyse de : The Eiffel Tower is in Paris.
🔎 Analyse de : it was built in 1990.
[REFUTED / UNVERIFIED] Tom Cruise is an american actor. (Source: https://www.wolframalpha.com)
[REFUTED / UNVERIFIED] The Eiffel Tower is in Paris. (Source: https://www.wolframalpha.com)
[REFUTED / UNVERIFIED] it was built in 1990. (Source: https://www.wolframalpha.com)
